# Notebook 01 - Data Contracts v1 POC


In [ ]:

# ============================================================
# Notebook 01 - Data Contracts
# Agent Evaluation Framework v1 POC
# ============================================================

try:
    run_id
except NameError:
    run_id = "RUN-MANUAL-TEST"

try:
    environment
except NameError:
    environment = "dev"

try:
    poc_mode
except NameError:
    poc_mode = "false"
poc_mode = str(poc_mode).lower()

import datetime as dt
from pathlib import Path

import requests
import yaml
from pyspark.sql import Row
from pyspark.sql.functions import col, max as spark_max
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, BooleanType

assert spark is not None, "Spark session not available."
from notebookutils import mssparkutils

LAKEHOUSE_NAME = "jacks_lakehouse"
ONELAKE_WORKSPACE_ID = "d9e51304-2b8a-4e62-b689-922e79fd76b4"
ONELAKE_LAKEHOUSE_ID = "537823c4-b83a-4a9b-9444-6039d55a4b9e"
LAKEHOUSE_DEFAULT_FILES_ROOT = "/lakehouse/default/Files"
LAKEHOUSE_FILES_ABFSS_ROOT = f"abfss://{ONELAKE_WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{ONELAKE_LAKEHOUSE_ID}/Files"
LAKEHOUSE_FILES_ROOT = LAKEHOUSE_FILES_ABFSS_ROOT
BASE_FILES_PATH = f"{LAKEHOUSE_FILES_ROOT}/agent_eval"
BASE_FILES_ABFSS_PATH = f"{LAKEHOUSE_FILES_ABFSS_ROOT}/agent_eval"
CONFIG_PATH = f"{BASE_FILES_PATH}/config"

AGENTS_YAML_PATH = f"{CONFIG_PATH}/agents.yaml"
TEST_CASES_CSV_PATH = f"{CONFIG_PATH}/test_cases.csv"
MICROSOFT_EVAL_TEST_SETS_YAML_PATH = f"{CONFIG_PATH}/microsoft_eval_test_sets.yaml"
FABRIC_DEPLOYMENT_MANIFEST_YAML_PATH = f"{CONFIG_PATH}/fabric_deployment_manifest.yaml"
JUDGE_CONFIG_YAML_PATH = f"{CONFIG_PATH}/judge_config.yaml"
CONTRACTS_TABLE = "agent_eval_data_contracts"

MAX_DATA_AGE_HOURS = 48
CRITICAL_SEVERITIES = {"critical"}


def now_utc():
    return dt.datetime.now(dt.timezone.utc)


def is_onelake_path(path):
    return str(path).startswith("abfss://")


def file_exists(path):
    if is_onelake_path(path):
        return mssparkutils.fs.exists(path)
    return Path(path).is_file()


def read_text(path, max_bytes=20 * 1024 * 1024):
    if is_onelake_path(path):
        return mssparkutils.fs.head(path, max_bytes)
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def load_agents_yaml(path):
    return (yaml.safe_load(read_text(path)) or {}).get("agents", [])


def pass_result(evidence, severity="major"):
    return {"passed": True, "severity": severity, "evidence": evidence}


def fail_result(evidence, severity="critical"):
    return {"passed": False, "severity": severity, "evidence": evidence}


def check_file_exists(path, severity="critical"):
    return pass_result(f"File exists: {path}", severity) if file_exists(path) else fail_result(f"Missing file: {path}", severity)


def check_fabric_table_exists(table_name):
    try:
        spark.sql(f"SELECT 1 FROM {table_name} LIMIT 1").collect()
        return pass_result(f"Table {table_name} is queryable", "critical")
    except Exception as exc:
        return fail_result(f"Table {table_name} is not queryable: {str(exc)[:250]}", "critical")


def check_fabric_table_has_recent_data(table_name, date_column="RunDate", max_age_hours=MAX_DATA_AGE_HOURS):
    try:
        df = spark.table(table_name)
        if date_column not in df.columns:
            return fail_result(f"{table_name} missing date column {date_column}", "critical")
        latest = df.agg(spark_max(col(date_column))).collect()[0][0]
        if latest is None:
            return fail_result(f"{table_name} has no latest {date_column}", "critical")
        latest_dt = latest if isinstance(latest, dt.datetime) else dt.datetime.combine(latest, dt.time())
        if latest_dt.tzinfo is None:
            latest_dt = latest_dt.replace(tzinfo=dt.timezone.utc)
        age_hours = (now_utc() - latest_dt).total_seconds() / 3600
        passed = age_hours <= max_age_hours
        evidence = f"Latest {date_column}: {latest_dt.isoformat()} ({age_hours:.1f}h old)"
        return pass_result(evidence, "critical") if passed else fail_result(evidence, "critical")
    except Exception as exc:
        return fail_result(f"Freshness check failed for {table_name}: {str(exc)[:250]}", "critical")


def check_fabric_table_has_required_columns(table_name, required_columns):
    try:
        existing = set(spark.table(table_name).columns)
        missing = [c for c in required_columns if c not in existing]
        if missing:
            return fail_result(f"{table_name} missing columns: {missing}", "critical")
        return pass_result(f"{table_name} has required columns: {required_columns}", "critical")
    except Exception as exc:
        return fail_result(f"Column check failed for {table_name}: {str(exc)[:250]}", "critical")


def check_url_reachable(url, timeout_seconds=10):
    try:
        response = requests.head(url, timeout=timeout_seconds, allow_redirects=True)
        if response.status_code in (405, 501):
            response = requests.get(url, timeout=timeout_seconds, stream=True)
            response.close()
        passed = 200 <= response.status_code < 300
        evidence = f"{url} returned HTTP {response.status_code}"
        return pass_result(evidence, "major") if passed else fail_result(evidence, "major")
    except Exception as exc:
        return fail_result(f"{url} unreachable: {str(exc)[:250]}", "major")


def check_test_cases_exist():
    return check_file_exists(TEST_CASES_CSV_PATH, "critical")


def check_ms_eval_config_exists():
    result = check_file_exists(MICROSOFT_EVAL_TEST_SETS_YAML_PATH, "critical")
    if not result["passed"]:
        return result
    data = yaml.safe_load(read_text(MICROSOFT_EVAL_TEST_SETS_YAML_PATH)) or {}
    if "microsoft_eval" not in data:
        return fail_result("microsoft_eval_test_sets.yaml missing microsoft_eval root", "critical")
    return pass_result("Microsoft Evaluation test-set registry is present", "critical")


def check_fabric_deployment_manifest_exists():
    result = check_file_exists(FABRIC_DEPLOYMENT_MANIFEST_YAML_PATH, "critical")
    if not result["passed"]:
        return result
    data = yaml.safe_load(read_text(FABRIC_DEPLOYMENT_MANIFEST_YAML_PATH)) or {}
    fabric = data.get("fabric") or {}
    notebook_items = fabric.get("notebook_items") or {}
    lakehouse_files = fabric.get("lakehouse_files") or {}
    if notebook_items.get("master") != "00_orchestrator":
        return fail_result("Fabric manifest master notebook must be 00_orchestrator", "critical")
    root = lakehouse_files.get("root", "")
    if not (root.startswith("/lakehouse/default/Files/") or root.startswith("abfss://")):
        return fail_result("Fabric manifest must use Lakehouse Files or OneLake ABFSS paths", "critical")
    if fabric.get("workspace_contract", {}).get("local_downloads_dependency") is not False:
        return fail_result("Fabric manifest must declare no local Downloads runtime dependency", "critical")
    return pass_result("Fabric deployment manifest is present and Fabric-native", "critical")


def check_judge_tunnel_reachable():
    result = check_file_exists(JUDGE_CONFIG_YAML_PATH, "critical")
    if not result["passed"]:
        return result
    judge = (yaml.safe_load(read_text(JUDGE_CONFIG_YAML_PATH)) or {}).get("judge") or {}
    base_url = (judge.get("base_url") or "").rstrip("/")
    model = judge.get("model") or ""
    if not base_url:
        return fail_result("judge_config.yaml missing judge.base_url", "critical")
    try:
        response = requests.get(f"{base_url}/models", timeout=15)
        response.raise_for_status()
        payload = response.json()
        model_names = [
            item.get("id") or item.get("name")
            for item in payload.get("data", [])
            if isinstance(item, dict)
        ]
        if model and model not in model_names:
            return fail_result(f"Judge tunnel reachable but model {model} not listed: {model_names}", "critical")
        return pass_result(f"Judge tunnel reachable at {base_url}; models={model_names}", "critical")
    except Exception as exc:
        return fail_result(f"Judge tunnel not reachable at {base_url}: {str(exc)[:250]}", "critical")


CONTRACT_REGISTRY = {
    "poc_config_files_exist": (check_file_exists, {"path": AGENTS_YAML_PATH, "severity": "critical"}),
    "poc_test_cases_exist": (check_test_cases_exist, {}),
    "poc_microsoft_eval_config_exists": (check_ms_eval_config_exists, {}),
    "poc_fabric_deployment_manifest_exists": (check_fabric_deployment_manifest_exists, {}),
    "poc_judge_tunnel_reachable": (check_judge_tunnel_reachable, {}),
}


def run_contract(agent, contract_name):
    if contract_name not in CONTRACT_REGISTRY:
        return fail_result(f"Contract {contract_name} is not registered", "critical")
    func, kwargs = CONTRACT_REGISTRY[contract_name]
    return func(**kwargs)


def row_for(agent_id, contract_name, result):
    return {
        "run_id": run_id,
        "agent_id": agent_id,
        "contract_name": contract_name,
        "passed": bool(result["passed"]),
        "severity": result["severity"],
        "evidence": result["evidence"],
        "checked_at": now_utc(),
    }


agents = [a for a in load_agents_yaml(AGENTS_YAML_PATH) if a.get("enabled") is True]
rows = []

for agent in agents:
    contracts = agent.get("data_contracts") or []
    if not contracts:
        rows.append(row_for(agent["agent_id"], "(none declared)", pass_result("No contracts declared", "info")))
        continue
    for contract_name in contracts:
        result = run_contract(agent, contract_name)
        rows.append(row_for(agent["agent_id"], contract_name, result))
        marker = "PASS" if result["passed"] else "FAIL"
        print(f"{marker} {agent['agent_id']} {contract_name}: {result['evidence']}")

schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("agent_id", StringType(), False),
    StructField("contract_name", StringType(), False),
    StructField("passed", BooleanType(), False),
    StructField("severity", StringType(), False),
    StructField("evidence", StringType(), True),
    StructField("checked_at", TimestampType(), False),
])

spark.createDataFrame([Row(**r) for r in rows], schema=schema).write.format("delta").mode("append").saveAsTable(CONTRACTS_TABLE)

if not rows:
    raise RuntimeError("Data contracts wrote no rows")

critical_failures = [r for r in rows if (not r["passed"]) and r["severity"] in CRITICAL_SEVERITIES]
print(f"Data contracts complete. rows={len(rows)} critical_failures={len(critical_failures)}")

try:
    from notebookutils import mssparkutils
    mssparkutils.notebook.exit("FAIL" if critical_failures else "PASS")
except ImportError:
    if critical_failures:
        raise RuntimeError("Critical data contract failure")
